In [1]:
# =============================================================================
# MODEL 6 — KERAS AUTOENCODER: CYBER ANOMALY / ZERO-DAY DETECTION
# Fixed version — resolves common Jupyter kernel crash causes:
#   1. TF session / GPU memory growth enabled before any model ops
#   2. .keras save format replaced with .h5 (broader version compat)
#   3. matplotlib fill_betweenx ylim bug fixed
#   4. Redundant DataFrame copies reduced to lower peak RAM
#   5. tf.random.set_seed moved after GPU config
# =============================================================================

# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings("ignore")

# ── CRITICAL: configure GPU memory growth BEFORE any TF import side-effects ──
# Failing to do this is the #1 cause of kernel OOM crashes with TF on GPU.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # Suppress C++ TF logs

import tensorflow as tf

# Allow GPU memory to grow incrementally instead of grabbing it all at once.
# Without this, TF pre-allocates the entire GPU VRAM → kernel crash on small GPUs.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass  # Must be set before GPUs are initialised; ignore if already done

# ── Reproducibility (set AFTER GPU config) ───────────────────────────────────
import numpy as np
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Numerical / data ─────────────────────────────────────────────────────────
import pandas as pd

# ── Plotting ─────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")          # Non-interactive backend — avoids Qt/Tk crashes
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

# ── Keras ────────────────────────────────────────────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks

# ── Persistence ──────────────────────────────────────────────────────────────
import joblib

print("TensorFlow version:", tf.__version__)
print("GPUs available    :", len(gpus))

TensorFlow version: 2.15.0
GPUs available    : 1


In [2]:
# =============================================================================
# SECTION 1 — CONFIGURATION
# =============================================================================

DATA_PATH        = "/Users/amitsingh/ML_Projects/WarfareAI/datasets/cybint_clean_train.csv"
MODEL_SAVE_PATH  = "model6_autoencoder.h5"    # FIX: .h5 is more stable across
                                               #      TF versions than .keras
SCALER_SAVE_PATH = "model6_scaler.pkl"
THRESH_SAVE_PATH = "model6_threshold.npy"

COUNT_COLS = [
    "n_intrusions",
    "n_jamming",
    "n_malware",
    "events_near_border",
    "n_active_scenarios",
]

CONTINUOUS_COLS = [
    "weighted_cyber_signal",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
]

FEATURES         = COUNT_COLS + CONTINUOUS_COLS
TARGET_COL       = "threat_label_int"
LOW_THREAT_LABEL = 0

INPUT_DIM        = len(FEATURES)   # 10
ENCODING_DIM     = 3
ENCODER_HIDDEN   = 6
DECODER_HIDDEN   = 6

EPOCHS            = 80
BATCH_SIZE        = 64
LEARNING_RATE     = 1e-3
VALIDATION_SPLIT  = 0.15
EARLY_STOP_PAT    = 10
THRESHOLD_PERCENTILE = 95


In [3]:
# =============================================================================
# SECTION 2 — DATA LOADING
# =============================================================================

print("\n" + "="*65)
print("SECTION 2 — LOADING & INSPECTING DATA")
print("="*65)

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {df.columns.tolist()}")
print(f"\nThreat label distribution:")
print(df[TARGET_COL].value_counts().sort_index().to_string())

missing = [f for f in FEATURES if f not in df.columns]
if missing:
    raise ValueError(f"Missing expected features in CSV: {missing}")
print(f"\nAll {len(FEATURES)} features present — OK")



SECTION 2 — LOADING & INSPECTING DATA
Dataset shape : (40000, 17)
Columns       : ['n_intrusions', 'n_jamming', 'n_malware', 'weighted_cyber_signal', 'events_near_border', 'n_active_scenarios', 'hour', 'day', 'month', 'dayofweek', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'region', 'region_code', 'threat_label_int']

Threat label distribution:
threat_label_int
0    11068
1    12015
2    10967
3     5950

All 10 features present — OK


In [5]:
# =============================================================================
# SECTION 3 — ISOLATE LOW-THREAT BASELINE
# =============================================================================

print("\n" + "="*65)
print("SECTION 3 — ISOLATING NORMAL BASELINE")
print("="*65)

normal_mask = df[TARGET_COL] == LOW_THREAT_LABEL

# FIX: Extract only the columns we need immediately; avoid holding full copies.
df_normal_feats  = df.loc[normal_mask, FEATURES].copy()
df_normal_feats[COUNT_COLS] = np.log1p(df_normal_feats[COUNT_COLS])   # log1p in-place

print(f"Normal rows : {normal_mask.sum():,}")
print(f"Attack rows : {(~normal_mask).sum():,}")


SECTION 3 — ISOLATING NORMAL BASELINE
Normal rows : 11,068
Attack rows : 28,932


In [6]:
# =============================================================================
# SECTION 4 — PREPROCESSING
# =============================================================================

print("\n" + "="*65)
print("SECTION 4 — PREPROCESSING")
print("="*65)

X_normal = df_normal_feats[FEATURES].values

X_train_raw, X_val_raw = train_test_split(
    X_normal, test_size=VALIDATION_SPLIT, random_state=SEED
)
print(f"Normal train rows : {len(X_train_raw):,}")
print(f"Normal val rows   : {len(X_val_raw):,}")

scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)

# Free the large intermediate array
del X_normal, X_train_raw, X_val_raw, df_normal_feats

print("\nScaler fitted. Feature ranges after scaling (train):")
for i, feat in enumerate(FEATURES):
    print(f"  {feat:<28} min={X_train[:,i].min():.3f}  max={X_train[:,i].max():.3f}")

joblib.dump(scaler, SCALER_SAVE_PATH)
print(f"\nScaler saved → {SCALER_SAVE_PATH}")


SECTION 4 — PREPROCESSING
Normal train rows : 9,407
Normal val rows   : 1,661

Scaler fitted. Feature ranges after scaling (train):
  n_intrusions                 min=0.000  max=1.000
  n_jamming                    min=0.000  max=1.000
  n_malware                    min=0.000  max=1.000
  events_near_border           min=0.000  max=1.000
  n_active_scenarios           min=0.000  max=0.000
  weighted_cyber_signal        min=0.000  max=1.000
  hour_sin                     min=0.000  max=1.000
  hour_cos                     min=0.000  max=1.000
  month_sin                    min=0.000  max=1.000
  month_cos                    min=0.000  max=1.000

Scaler saved → model6_scaler.pkl


In [7]:
# =============================================================================
# SECTION 5 — BUILD THE AUTOENCODER
# =============================================================================

print("\n" + "="*65)
print("SECTION 5 — BUILDING AUTOENCODER")
print("="*65)

def build_autoencoder(input_dim, encoding_dim, encoder_hidden,
                      decoder_hidden, learning_rate):
    inputs     = keras.Input(shape=(input_dim,), name="input_layer")

    # Encoder
    enc        = layers.Dense(encoder_hidden, name="enc_hidden")(inputs)
    enc        = layers.BatchNormalization(name="enc_bn")(enc)
    enc        = layers.Activation("relu", name="enc_relu")(enc)
    bottleneck = layers.Dense(encoding_dim, activation="relu",
                              name="bottleneck")(enc)

    # Decoder
    dec        = layers.Dense(decoder_hidden, name="dec_hidden")(bottleneck)
    dec        = layers.BatchNormalization(name="dec_bn")(dec)
    dec        = layers.Activation("relu", name="dec_relu")(dec)
    outputs    = layers.Dense(input_dim, activation="sigmoid",
                              name="reconstruction")(dec)

    model = Model(inputs=inputs, outputs=outputs, name="CyberAutoencoder")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae"],
    )
    return model


autoencoder = build_autoencoder(
    INPUT_DIM, ENCODING_DIM, ENCODER_HIDDEN, DECODER_HIDDEN, LEARNING_RATE
)
autoencoder.summary()


SECTION 5 — BUILDING AUTOENCODER


Model: "CyberAutoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_hidden (Dense)              │ (None, 6)              │            66 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_bn (BatchNormalization)     │ (None, 6)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_relu (Activation)           │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 3)              │            21 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_hidden (Dense)              │ (None, 6)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_bn (BatchNormalization)     │ (None, 6)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_relu (Activation)           │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction (Dense)          │ (None, 10)             │            70 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 229 (916.00 B)

 Trainable params: 205 (820.00 B)

 Non-trainable params: 24 (96.00 B)

In [8]:
# =============================================================================
# SECTION 6 — TRAINING
# =============================================================================

print("\n" + "="*65)
print("SECTION 6 — TRAINING")
print("="*65)

early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=EARLY_STOP_PAT,
    restore_best_weights=True, verbose=1,
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=5,
    min_lr=1e-6, verbose=1,
)

history = autoencoder.fit(
    X_train, X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, X_val),
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)

print(f"\nStopped at epoch : {len(history.history['loss'])}")
print(f"Best val_loss    : {min(history.history['val_loss']):.6f}")

# FIX: use .h5 format — the native .keras format can trigger serialisation
#      errors in older TF builds that crash the kernel on save.
autoencoder.save(MODEL_SAVE_PATH)
print(f"Model saved → {MODEL_SAVE_PATH}")




SECTION 6 — TRAINING
Epoch 1/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - loss: 0.1263 - mae: 0.3106 - val_loss: 0.1096 - val_mae: 0.2932 - learning_rate: 0.0010
Epoch 2/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0839 - mae: 0.2499 - val_loss: 0.0735 - val_mae: 0.2337 - learning_rate: 0.0010
Epoch 3/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0637 - mae: 0.2110 - val_loss: 0.0601 - val_mae: 0.2033 - learning_rate: 0.0010
Epoch 4/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0556 - mae: 0.1921 - val_loss: 0.0537 - val_mae: 0.1873 - learning_rate: 0.0010
Epoch 5/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0515 - mae: 0.1823 - val_loss: 0.0493 - val_mae: 0.1763 - learning_rate: 0.0010
Epoch 6/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0484 - mae: 0.1756 - val_loss: 0.0463 - val_mae: 0.1694 - learning_rate: 0.0010
Epoch 7/80
147/147 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0465 - mae: 0.1707 - val_loss: 0.0448 - val_mae: 0.1658 -


Stopped at epoch : 21
Best val_loss    : 0.042126
Model saved → model6_autoencoder.h5


In [9]:
# =============================================================================
# SECTION 7 — ANOMALY THRESHOLD
# =============================================================================

print("\n" + "="*65)
print("SECTION 7 — COMPUTING ANOMALY THRESHOLD")
print("="*65)

X_train_pred = autoencoder.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
train_mse    = np.mean(np.square(X_train - X_train_pred), axis=1)

print(f"Training MSE — mean={train_mse.mean():.6f}  std={train_mse.std():.6f}"
      f"  min={train_mse.min():.6f}  max={train_mse.max():.6f}")

THRESHOLD = float(np.percentile(train_mse, THRESHOLD_PERCENTILE))
print(f"Threshold ({THRESHOLD_PERCENTILE}th pct) = {THRESHOLD:.6f}")

np.save(THRESH_SAVE_PATH, np.array(THRESHOLD))
print(f"Threshold saved → {THRESH_SAVE_PATH}")




SECTION 7 — COMPUTING ANOMALY THRESHOLD
Training MSE — mean=0.042650  std=0.022978  min=0.003134  max=0.166380
Threshold (95th pct) = 0.084596
Threshold saved → model6_threshold.npy


In [10]:
# =============================================================================
# SECTION 8 — FULL-DATASET EVALUATION
# FIX: process in chunks if memory is tight; use a single transform call.
# =============================================================================

print("\n" + "="*65)
print("SECTION 8 — FULL-DATASET INFERENCE & EVALUATION")
print("="*65)

# Build scaled feature matrix for the full dataset
df_all_feats = df[FEATURES].copy()
df_all_feats[COUNT_COLS] = np.log1p(df_all_feats[COUNT_COLS])
X_all_scaled = scaler.transform(df_all_feats[FEATURES].values)
del df_all_feats   # Free memory before prediction

X_all_pred    = autoencoder.predict(X_all_scaled, batch_size=BATCH_SIZE, verbose=0)
all_mse       = np.mean(np.square(X_all_scaled - X_all_pred), axis=1)
y_pred_binary = (all_mse > THRESHOLD).astype(int)
y_true_binary = (df[TARGET_COL] != LOW_THREAT_LABEL).astype(int).values

print("\nClassification Report:")
print(classification_report(
    y_true_binary, y_pred_binary,
    target_names=["Normal", "Anomaly"], digits=4,
))

auc = roc_auc_score(y_true_binary, all_mse)
print(f"ROC-AUC : {auc:.4f}")

df_eval = pd.DataFrame({
    TARGET_COL : df[TARGET_COL].values,
    "mse"      : all_mse,
    "anomaly"  : y_pred_binary,
})

print("\nMean MSE per threat level:")
print(df_eval.groupby(TARGET_COL)["mse"].agg(["mean","std","min","max"]).round(6))

print("\nDetection rate per threat level:")
for label, rate in df_eval.groupby(TARGET_COL)["anomaly"].mean().items():
    lname = "LOW" if label == 0 else f"THREAT-{label}"
    print(f"  Label {label} ({lname:<10}) : {rate:.1%} flagged")



SECTION 8 — FULL-DATASET INFERENCE & EVALUATION

Classification Report:
              precision    recall  f1-score   support

      Normal     0.8375    0.9497    0.8900     11068
     Anomaly     0.9797    0.9295    0.9539     28932

    accuracy                         0.9351     40000
   macro avg     0.9086    0.9396    0.9220     40000
weighted avg     0.9403    0.9351    0.9363     40000

ROC-AUC : 0.9798

Mean MSE per threat level:
                      mean       std       min       max
threat_label_int                                        
0                 0.042571  0.023100  0.003134  0.166380
1                 0.148416  0.064305  0.009213  0.422952
2                 0.346240  0.093081  0.056500  0.653100
3                 0.579592  0.091307  0.246669  0.806215

Detection rate per threat level:
  Label 0 (LOW       ) : 5.0% flagged
  Label 1 (THREAT-1  ) : 83.1% flagged
  Label 2 (THREAT-2  ) : 99.9% flagged
  Label 3 (THREAT-3  ) : 100.0% flagged


In [11]:
# =============================================================================
# SECTION 9 — PER-ATTACK-LEVEL BREAKDOWN
# =============================================================================

print("\n" + "="*65)
print("SECTION 9 — ATTACK LEVEL BREAKDOWN")
print("="*65)

label_map = {0:"LOW (normal)", 1:"MED attack", 2:"HIGH attack", 3:"CRIT attack"}
for lvl in sorted(df[TARGET_COL].unique()):
    mask   = df[TARGET_COL].values == lvl
    n_tot  = mask.sum()
    n_flag = y_pred_binary[mask].sum()
    print(f"  {label_map.get(lvl, f'label={lvl}'):<20} "
          f"total={n_tot:5,}  flagged={n_flag:5,}  "
          f"detection={n_flag/n_tot:.1%}")




SECTION 9 — ATTACK LEVEL BREAKDOWN
  LOW (normal)         total=11,068  flagged=  557  detection=5.0%
  MED attack           total=12,015  flagged=9,983  detection=83.1%
  HIGH attack          total=10,967  flagged=10,959  detection=99.9%
  CRIT attack          total=5,950  flagged=5,950  detection=100.0%


In [12]:
# =============================================================================
# SECTION 10 — VISUALISATIONS
# FIX: fill_betweenx ylim crash — draw histogram first, retrieve ylim after.
# =============================================================================

print("\n" + "="*65)
print("SECTION 10 — PLOTTING DIAGNOSTICS")
print("="*65)

DARK_BG = "#0d1117"
PANEL   = "#161b22"
ACCENT  = "#58a6ff"
WARN    = "#f85149"
OK      = "#3fb950"
GRID    = "#21262d"
TEXT    = "#c9d1d9"

plt.rcParams.update({
    "axes.facecolor"  : PANEL,
    "axes.edgecolor"  : GRID,
    "axes.labelcolor" : TEXT,
    "xtick.color"     : TEXT,
    "ytick.color"     : TEXT,
    "text.color"      : TEXT,
    "grid.color"      : GRID,
    "figure.facecolor": DARK_BG,
})

fig = plt.figure(figsize=(20, 18), facecolor=DARK_BG)
fig.suptitle("Model 6 — Keras Autoencoder: Cyber Anomaly Detection",
             color="white", fontsize=18, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)


# ── [A] Training curves ────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ep  = range(1, len(history.history["loss"]) + 1)
ax1.plot(ep, history.history["loss"],     color=ACCENT, lw=2, label="Train MSE")
ax1.plot(ep, history.history["val_loss"], color=WARN,   lw=2, linestyle="--",
         label="Val MSE")
ax1.set_title("A — Training & Validation Loss", color=TEXT, pad=8)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE Loss")
ax1.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT)
ax1.grid(True, alpha=0.4)


# ── [B] MSE violin per threat level ───────────────────────────────────────
ax2    = fig.add_subplot(gs[0, 1])
levels = sorted(df[TARGET_COL].unique())
vdata  = [df_eval[df_eval[TARGET_COL] == lvl]["mse"].values for lvl in levels]
parts  = ax2.violinplot(vdata, positions=levels, showmedians=True, showextrema=True)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor([OK, "#e3b341", WARN, "#f85149"][i])
    pc.set_alpha(0.7)
for elem in ["cbars","cmins","cmaxes","cmedians"]:
    parts[elem].set_color("white")
ax2.axhline(THRESHOLD, color=WARN, linestyle="--", lw=1.5,
            label=f"Threshold ({THRESHOLD:.4f})")
ax2.set_xticks(levels)
ax2.set_xticklabels(["LOW\n(0)","MED\n(1)","HIGH\n(2)","CRIT\n(3)"])
ax2.set_title("B — MSE Distribution by Threat Level", color=TEXT, pad=8)
ax2.set_ylabel("Reconstruction MSE")
ax2.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT)
ax2.grid(True, alpha=0.4)


# ── [C] MSE histogram + threshold  (FIX: read ylim AFTER drawing hist) ────
ax3        = fig.add_subplot(gs[1, 0])
normal_mse = df_eval[df_eval[TARGET_COL] == 0]["mse"].values
ax3.hist(normal_mse, bins=60, color=ACCENT, alpha=0.75,
         label="Normal rows MSE", edgecolor="none")

# FIX: retrieve ylim AFTER the histogram is drawn, not before.
y_lo, y_hi = ax3.get_ylim()
ax3.axvline(THRESHOLD, color=WARN, lw=2, linestyle="--",
            label=f"95th pct = {THRESHOLD:.4f}")
ax3.fill_betweenx(
    [y_lo, y_hi],
    THRESHOLD, normal_mse.max() * 1.05,
    color=WARN, alpha=0.08, label="Anomaly zone",
)
ax3.set_ylim(y_lo, y_hi)   # Restore so fill doesn't resize the axes
ax3.set_title("C — Normal-Row MSE Histogram + Threshold", color=TEXT, pad=8)
ax3.set_xlabel("Reconstruction MSE"); ax3.set_ylabel("Count")
ax3.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT)
ax3.grid(True, alpha=0.4)


# ── [D] Confusion matrix ──────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
cm  = confusion_matrix(y_true_binary, y_pred_binary)
sns.heatmap(
    cm, annot=True, fmt="d", ax=ax4, cmap="Blues",
    xticklabels=["Pred Normal","Pred Anomaly"],
    yticklabels=["True Normal","True Anomaly"],
    annot_kws={"size":14, "color":"white"},
    linecolor=DARK_BG, linewidths=0.5,
)
ax4.set_title("D — Confusion Matrix", color=TEXT, pad=8)
ax4.set_xlabel("Predicted"); ax4.set_ylabel("Actual")


# ── [E] ROC curve ─────────────────────────────────────────────────────────
ax5       = fig.add_subplot(gs[2, 0])
fpr, tpr, _ = roc_curve(y_true_binary, all_mse)
ax5.plot(fpr, tpr, color=ACCENT, lw=2, label=f"AUC = {auc:.4f}")
ax5.plot([0,1],[0,1], color=GRID, linestyle="--", lw=1)
ax5.fill_between(fpr, tpr, alpha=0.15, color=ACCENT)
ax5.set_title("E — ROC Curve", color=TEXT, pad=8)
ax5.set_xlabel("False Positive Rate"); ax5.set_ylabel("True Positive Rate")
ax5.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT)
ax5.grid(True, alpha=0.4)


# ── [F] Per-feature mean abs error ────────────────────────────────────────
ax6         = fig.add_subplot(gs[2, 1])
normal_mask = df[TARGET_COL].values == 0
crit_mask   = df[TARGET_COL].values == 3

feat_err_n = np.abs(X_all_scaled[normal_mask] - X_all_pred[normal_mask]).mean(axis=0)
feat_err_c = np.abs(X_all_scaled[crit_mask]   - X_all_pred[crit_mask]  ).mean(axis=0)

x_pos = np.arange(INPUT_DIM); w = 0.38
ax6.bar(x_pos - w/2, feat_err_n, width=w, color=OK,   alpha=0.8, label="Normal (0)")
ax6.bar(x_pos + w/2, feat_err_c, width=w, color=WARN, alpha=0.8, label="CRIT (3)")
ax6.set_xticks(x_pos)
ax6.set_xticklabels(FEATURES, rotation=45, ha="right", fontsize=8)
ax6.set_title("F — Per-Feature MAE: Normal vs CRIT", color=TEXT, pad=8)
ax6.set_ylabel("Mean |x − x̂|")
ax6.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT)
ax6.grid(True, alpha=0.4, axis="y")


plt.savefig("model6_diagnostics.png", dpi=150, bbox_inches="tight",
            facecolor=DARK_BG)
print("Diagnostic plot saved → model6_diagnostics.png")
plt.close(fig)   # FIX: explicitly close to free figure memory


SECTION 10 — PLOTTING DIAGNOSTICS
Diagnostic plot saved → model6_diagnostics.png


In [26]:
# =============================================================================
# SECTION 11 — INFERENCE FUNCTION
# =============================================================================

print("\n" + "="*65)
print("SECTION 11 — INFERENCE FUNCTION DEMO")
print("="*65)

def predict_anomaly(raw_df, model_path=MODEL_SAVE_PATH,
                    scaler_path=SCALER_SAVE_PATH, thresh_path=THRESH_SAVE_PATH):
    """
    Score new rows for zero-day / novel attack detection.

    Parameters
    ----------
    raw_df      : DataFrame with at least the 10 FEATURES columns (raw values).
    model_path  : Path to saved .h5 model.
    scaler_path : Path to saved joblib scaler.
    thresh_path : Path to saved .npy threshold.

    Returns
    -------
    DataFrame with mse_score and is_anomaly columns appended.
    """
    model_     = keras.models.load_model(model_path, custom_objects={'mse': tf.keras.metrics.MeanSquaredError})
    scaler_    = joblib.load(scaler_path)
    threshold  = float(np.load(thresh_path))

    df_proc = raw_df[FEATURES].copy()
    df_proc[COUNT_COLS] = np.log1p(df_proc[COUNT_COLS])

    X_scaled = scaler_.transform(df_proc.values)
    X_recon  = model_.predict(X_scaled, batch_size=256, verbose=0)
    mse      = np.mean(np.square(X_scaled - X_recon), axis=1)

    result               = raw_df.copy()
    result["mse_score"]  = mse
    result["is_anomaly"] = (mse > threshold).astype(int)
    return result


sample_rows = df.sample(5, random_state=99)
scored      = predict_anomaly(sample_rows)
print("Demo — 5 random rows scored:")
print(scored[[TARGET_COL, "mse_score", "is_anomaly"]].to_string(index=False))


SECTION 11 — INFERENCE FUNCTION DEMO
Demo — 5 random rows scored:
 threat_label_int  mse_score  is_anomaly
                0   0.028552           0
                1   0.166834           1
                3   0.708137           1
                1   0.166586           1
                2   0.332591           1


In [27]:
# =============================================================================
# SECTION 12 — SUMMARY
# =============================================================================

print("\n" + "="*65)
print("SECTION 12 — FINAL SUMMARY")
print("="*65)

n_attack   = y_true_binary.sum()
n_detected = ((y_pred_binary == 1) & (y_true_binary == 1)).sum()
n_fp       = ((y_pred_binary == 1) & (y_true_binary == 0)).sum()
n_normal   = (y_true_binary == 0).sum()

print(f"  Total rows evaluated       : {len(y_true_binary):,}")
print(f"  True attack rows           : {n_attack:,}")
print(f"  Correctly detected attacks : {n_detected:,}  ({n_detected/n_attack:.1%} recall)")
print(f"  False positives            : {n_fp:,}  ({n_fp/n_normal:.1%} of normal rows)")
print(f"  ROC-AUC                    : {auc:.4f}")
print(f"  Anomaly threshold (95th)   : {THRESHOLD:.6f}")
print(f"\nSaved artefacts:")
for p in [MODEL_SAVE_PATH, SCALER_SAVE_PATH, THRESH_SAVE_PATH, "model6_diagnostics.png"]:
    print(f"  {p}")
print("\nDone.")


SECTION 12 — FINAL SUMMARY
  Total rows evaluated       : 40,000
  True attack rows           : 28,932
  Correctly detected attacks : 26,892  (92.9% recall)
  False positives            : 557  (5.0% of normal rows)
  ROC-AUC                    : 0.9798
  Anomaly threshold (95th)   : 0.084596

Saved artefacts:
  model6_autoencoder.h5
  model6_scaler.pkl
  model6_threshold.npy
  model6_diagnostics.png

Done.
